# South Asian Summer Monsoon JJA Precipitation — Time of Emergence
## G6-1.5K-SAI and G6-1.5K-HiLLA vs SSP2-4.5 | All four models

**Variable:** JJA-mean, area-weighted precipitation over the Nalam et al. (2018) South Asian
monsoon box (5–40°N, 60–100°E), in mm/day (absolute values). As defined in `ToE_testing_v9`.

**Method (as in `SeaIce_ToE_ttest`):** Cumulative one-sided Welch's t-test applied to each paired
member, testing whether monsoon precipitation under the geoengineering scenario is *less than*
that of its paired SSP2-4.5 member (i.e. monsoon weakening under aerosol forcing relative to the
counterfactual), p < 0.05. **Member ToE** = first year Y ≥ 2035 at which the cumulative window
[2035, Y] is significant. **Model ToE ("p66")** = k-th earliest member ToE, k = ⌈0.66 × N⌉.

| Model | Members | SAI precip |
|---|---|---|
| MIROC-ES2H | 10 | yes (check path) |
| CESM2-WACCM | 3 | yes |
| UKESM1-1 | 3 | no |
| E3SMv3 | 3 | no |

CESM2-WACCM SSP2-4.5 uses the first 3 members of the public ARISE ensemble (r001–r003), paired 1:1
with HiLLA/SAI r1–r3. MIROC-ES2H bootstrap: all C(10,3) = 120 three-member subsets.

Precipitation variables/units: `PRECT` (m s⁻¹, × 8.64×10⁷ → mm/day) for CESM2-WACCM and E3SMv3;
`pr` (kg m⁻² s⁻¹, × 86400 → mm/day) for UKESM1-1 and MIROC-ES2H.

In [1]:
import sys
import fsspec
import xarray as xr
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker
from scipy import stats
from itertools import combinations
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

matplotlib.rcParams.update({'font.size': 15})

sys.path.append('../shared/Code_examples/G6-1.5K_example/')
from Utils import set_time_to_center_of_bounds

SAI_START  = 2035
ALPHA      = 0.05
MIN_N      = 2
PLOT_START = 2020
PLOT_END   = 2084

# Nalam et al. 2018 South Asian monsoon box (as in ToE_testing_v9)
MON_LAT = (5, 40)
MON_LON = (60, 100)

# unit conversions to mm/day
MPS_TO_MMDAY   = 86400.0 * 1000.0   # PRECT: m/s      -> mm/day
KGM2S_TO_MMDAY = 86400.0            # pr:    kg/m2/s  -> mm/day

## 1. File paths

In [2]:
# ── CESM2-WACCM ───────────────────────────────────────────────────────────────
# SSP2-4.5: ARISE public bucket (anon=True); members 001-005 run to 2100, 006-010 to 2069.
# We load only the first 3 members for pairing with HiLLA/SAI r1-r3.
ssp245_base   = 'ncar-cesm2-arise/CESM2-WACCM-SSP245/'
_ssp245_ends  = {m: '210012' if int(m) <= 5 else '206912'
                 for m in [f'{i:03d}' for i in range(1, 11)]}
cesm_ssp245_pr_files = [
    [f's3://{ssp245_base}b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.{m}/atm/proc/tseries/month_1/'
     f'b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.{m}.cam.h0.PRECT.201501-206412.nc',
     f's3://{ssp245_base}b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.{m}/atm/proc/tseries/month_1/'
     f'b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.{m}.cam.h0.PRECT.206501-{_ssp245_ends[m]}.nc']
    for m in list(_ssp245_ends)[:3]   # paired members only
]
cesm_HiLLA_pr_files = [
    ['s3://reflective-persistent-prod-large/CESM2-WACCM/G6-1.5k-HiLLA/r1/AMON/b.e21.BW.f09_g17.SSP245-G6-1p5K-HiLLA.001.cam.h0.PRECT.203501-203912.nc',
     's3://reflective-persistent-prod-large/CESM2-WACCM/G6-1.5k-HiLLA/r1/AMON/b.e21.BW.f09_g17.SSP245-G6-1p5K-HiLLA.001.cam.h0.PRECT.204001-208412.nc'],
    ['s3://reflective-persistent-prod-large/CESM2-WACCM/G6-1.5k-HiLLA/r2/AMON/b.e21.BW.f09_g17.SSP245-G6-1p5K-HiLLA.002.cam.h0.PRECT.203501-208412.nc'],
    ['s3://reflective-persistent-prod-large/CESM2-WACCM/G6-1.5k-HiLLA/r3/AMON/b.e21.BW.f09_g17.SSP245-G6-1p5K-HiLLA.003.cam.h0.PRECT.203501-208412.nc'],
]
cesm_SAI_pr_files = [
    ['s3://reflective-persistent-prod-large/CESM2-WACCM/G6-1.5k-SAI/r1/AMON/b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.001.cam.h0.PRECT.203501-208412.nc'],
    ['s3://reflective-persistent-prod-large/CESM2-WACCM/G6-1.5k-SAI/r2/AMON/b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.002.cam.h0.PRECT.203501-208412.nc'],
    ['s3://reflective-persistent-prod-large/CESM2-WACCM/G6-1.5k-SAI/r3/AMON/b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.003.cam.h0.PRECT.203501-208412.nc'],
]

# ── UKESM1-1 — SAI precip not available ───────────────────────────────────────
# NOTE: directory guess — sea ice lived in apm/SImon/siconca (HiLLA) and ap5/SImon/siconca
# (SSP245); assuming the Amon/pr equivalent. Run the path-check cell below to verify.
ukesm_HiLLA_pr_paths = [
    's3://reflective-persistent-prod-large/UKESM1-1/G6-1p5K-HiLLA/r12i1p1f2/apm/Amon/pr/',
    's3://reflective-persistent-prod-large/UKESM1-1/G6-1p5K-HiLLA/r2i1p1f2/apm/Amon/pr/',
    's3://reflective-persistent-prod-large/UKESM1-1/G6-1p5K-HiLLA/r3i1p1f2/apm/Amon/pr/',
]
ukesm_ssp245_pr_paths = [
    's3://reflective-persistent-prod-large/UKESM1-1/SSP245/r12i1p1f1/ap5/Amon/pr/',
    's3://reflective-persistent-prod-large/UKESM1-1/SSP245/r2i1p1f1/ap5/Amon/pr/',
    's3://reflective-persistent-prod-large/UKESM1-1/SSP245/r3i1p1f1/ap5/Amon/pr/',
]

# ── MIROC-ES2H — all 10 members ───────────────────────────────────────────────
_MIROC = 's3://reflective-persistent-prod-large/MIROC-ES2H'
miroc_HiLLA_pr_files  = [f'{_MIROC}/G6-1.5K-HiLLA/Amon/pr_G6-1.5K-HiLLA_r{i:02d}.nc' for i in range(1, 11)]
miroc_ssp245_pr_files = [f'{_MIROC}/G6-1.5K-HiLLA/Amon/pr_baseline_r{i:02d}.nc'       for i in range(1, 11)]
# NOTE: guess — SAI sea ice lived in G6-1.5K-SAI/Mon/SeaIce_...; assuming pr sits alongside.
# The path-check cell will list the SAI directory if this is wrong.
miroc_SAI_pr_files    = [f'{_MIROC}/G6-1.5K-SAI/Mon/P_G6-1.5K-SAI_r{i:02d}.nc'       for i in range(1, 11)]

# ── E3SMv3 — SAI precip not available; HiLLA/SSP245 already regridded ─────────
_E3SM_DEST = 'reflective-persistent-prod'
e3sm_HiLLA_pr_files  = [[f's3://{_E3SM_DEST}/alistairduffey/E3SMv3/G6-1.5K-HiLLA/{r}/PRECT/PRECT_combined.nc'] for r in ['r1', 'r2', 'r3']]
e3sm_ssp245_pr_files = [[f's3://{_E3SM_DEST}/alistairduffey/E3SMv3/SSP245/{r}/PRECT/PRECT_combined.nc']         for r in ['r1', 'r2', 'r3']]

print('File paths defined.')

File paths defined.


## 2. Path verification

Several paths above are inferred from the sea-ice notebook by swapping the variable
(marked `NOTE: guess`). This cell checks every path before any data is loaded, and lists
the parent directory for anything missing so the correct path is easy to spot.

In [3]:
import s3fs

fs      = s3fs.S3FileSystem()
fs_anon = s3fs.S3FileSystem(anon=True)

def _strip(p):
    return p.replace('s3://', '')

def check_files(label, file_lists, anon=False):
    f = fs_anon if anon else fs
    missing = []
    flat = [p for sub in file_lists for p in (sub if isinstance(sub, list) else [sub])]
    for p in flat:
        if not f.exists(_strip(p)):
            missing.append(p)
    status = 'OK' if not missing else f'{len(missing)}/{len(flat)} MISSING'
    print(f'{label:<28} {status}')
    for p in missing:
        parent = _strip(p).rsplit('/', 1)[0]
        try:
            listing = f.ls(parent)
            print(f'   missing: {p}')
            print(f'   parent dir contains: {[x.rsplit("/",1)[-1] for x in listing][:12]}')
        except Exception as e:
            print(f'   missing: {p}  (parent unlistable: {e})')
    return not missing

def check_dirs(label, dir_paths, anon=False):
    f = fs_anon if anon else fs
    ok = True
    for p in dir_paths:
        try:
            n = len(f.glob(_strip(p) + '*.nc'))
        except Exception:
            n = 0
        if n == 0:
            ok = False
            grand = _strip(p).rstrip('/').rsplit('/', 1)[0]
            try:
                listing = f.ls(grand)
                print(f'{label}: no .nc files in {p}')
                print(f'   parent dir contains: {[x.rsplit("/",1)[-1] for x in listing][:12]}')
            except Exception as e:
                print(f'{label}: no .nc files in {p} (parent unlistable: {e})')
    if ok:
        print(f'{label:<28} OK')
    return ok

all_ok = True
all_ok &= check_files('CESM SSP2-4.5 (ARISE, anon)', cesm_ssp245_pr_files, anon=True)
all_ok &= check_files('CESM G6-1.5K-HiLLA',          cesm_HiLLA_pr_files)
all_ok &= check_files('CESM G6-1.5K-SAI',            cesm_SAI_pr_files)
all_ok &= check_dirs ('UKESM G6-1.5K-HiLLA',         ukesm_HiLLA_pr_paths)
all_ok &= check_dirs ('UKESM SSP2-4.5',              ukesm_ssp245_pr_paths)
all_ok &= check_files('MIROC G6-1.5K-HiLLA',         miroc_HiLLA_pr_files)
all_ok &= check_files('MIROC SSP2-4.5',              miroc_ssp245_pr_files)
all_ok &= check_files('MIROC G6-1.5K-SAI',           miroc_SAI_pr_files)
all_ok &= check_files('E3SM G6-1.5K-HiLLA',          e3sm_HiLLA_pr_files)
all_ok &= check_files('E3SM SSP2-4.5',               e3sm_ssp245_pr_files)
print('\nAll paths verified.' if all_ok else '\nFix the paths above before proceeding.')

CESM SSP2-4.5 (ARISE, anon)  OK
CESM G6-1.5K-HiLLA           OK
CESM G6-1.5K-SAI             OK
UKESM G6-1.5K-HiLLA          OK
UKESM SSP2-4.5               OK
MIROC G6-1.5K-HiLLA          OK
MIROC SSP2-4.5               OK
MIROC G6-1.5K-SAI            10/10 MISSING
   missing: s3://reflective-persistent-prod-large/MIROC-ES2H/G6-1.5K-SAI/Mon/pr_G6-1.5K-SAI_r01.nc
   parent dir contains: ['AOD550nm_G6-1.5K-SAI_r01.nc', 'AOD550nm_G6-1.5K-SAI_r02.nc', 'AOD550nm_G6-1.5K-SAI_r03.nc', 'AOD550nm_G6-1.5K-SAI_r04.nc', 'AOD550nm_G6-1.5K-SAI_r05.nc', 'AOD550nm_G6-1.5K-SAI_r06.nc', 'AOD550nm_G6-1.5K-SAI_r07.nc', 'AOD550nm_G6-1.5K-SAI_r08.nc', 'AOD550nm_G6-1.5K-SAI_r09.nc', 'AOD550nm_G6-1.5K-SAI_r10.nc', 'AOD550nm_baseline_r01.nc', 'AOD550nm_baseline_r02.nc']
   missing: s3://reflective-persistent-prod-large/MIROC-ES2H/G6-1.5K-SAI/Mon/pr_G6-1.5K-SAI_r02.nc
   parent dir contains: ['AOD550nm_G6-1.5K-SAI_r01.nc', 'AOD550nm_G6-1.5K-SAI_r02.nc', 'AOD550nm_G6-1.5K-SAI_r03.nc', 'AOD550nm_G6-1.5K-SAI_r04.

## 3. Reader functions and JJA monsoon computation

In [4]:
def read_pr_from_file_list(file_list_of_lists, var='PRECT', open_kwargs=None):
    """Read precip data for multiple members, each given as a list of file paths
    (CESM2-WACCM, E3SMv3). Renames `var` to 'pr'; unit conversion happens later."""
    if open_kwargs is None:
        open_kwargs = {}
    ds_list = []
    for member_files in file_list_of_lists:
        file_objects = [fsspec.open(f, mode='rb', **open_kwargs) for f in member_files]
        ds = xr.open_mfdataset([f.open() for f in file_objects],
                               combine='nested', concat_dim='time', chunks={}).load()
        _, idx = np.unique(ds.time, return_index=True)
        ds = ds.isel(time=idx)
        if var in ds and var != 'pr':
            ds = ds.rename({var: 'pr'})
        ds_list.append(ds)
    return ds_list


def read_pr_from_directory_paths(path_list, time_slice=None):
    """Read precip data where each member lives in a directory of .nc files (UKESM)."""
    ds_list = []
    for path in path_list:
        files = fsspec.open_files(f'{path}*.nc', mode='rb')
        ds = xr.open_mfdataset([f.open() for f in files],
                               combine='nested', concat_dim='time', chunks={}).load()
        _, idx = np.unique(ds.time, return_index=True)
        ds = ds.isel(time=idx)
        if time_slice:
            ds = ds.sel(time=slice(*time_slice))
        ds_list.append(ds)
    return ds_list


def open_nc(path):
    """Single-file reader (MIROC-ES2H Amon files)."""
    with fsspec.open(path, mode='rb') as f:
        return xr.open_dataset(f, engine='h5netcdf', chunks={}).load()


def read_miroc_pr(file_list):
    """MIROC-ES2H precip: standardise the variable name to 'pr' if needed."""
    ds_list = []
    for f in file_list:
        ds = open_nc(f)
        if 'pr' not in ds:
            pr_var = next((v for v in ds.data_vars
                           if 'pr' in v.lower() or 'precip' in v.lower()), None)
            if pr_var:
                ds = ds.rename({pr_var: 'pr'})
        ds_list.append(ds)
    return ds_list


# ── JJA monsoon-box mean (Nalam et al. 2018 box, as in ToE_testing_v9) ────────
def jja_monsoon_ts(ds_list, to_mm_day, var='pr', lat_nm='lat', lon_nm='lon'):
    """Area-weighted JJA-mean precip (mm/day) over MON_LAT x MON_LON, one
    DataArray per member, indexed by integer year. Years with an incomplete
    JJA (< 3 months) are dropped."""
    out = []
    for ds in ds_list:
        da = ds[var]
        if 'latitude' in da.dims:
            da = da.rename({'latitude': 'lat', 'longitude': 'lon'})
        if float(da[lat_nm][0]) > float(da[lat_nm][-1]):
            da = da.sortby(lat_nm)
        da_box  = da.sel(**{lat_nm: slice(*MON_LAT), lon_nm: slice(*MON_LON)})
        weights = np.cos(np.deg2rad(da_box[lat_nm]))
        weights = weights / weights.sum()
        sp_mean = (da_box * weights).sum(lat_nm).mean(lon_nm)
        jja     = sp_mean.sel(time=sp_mean.time.dt.month.isin([6, 7, 8]))
        cnt     = jja.groupby('time.year').count('time')
        annual  = jja.groupby('time.year').mean('time') * to_mm_day
        annual  = annual.where(cnt == 3, drop=True)
        annual  = annual.rename({'year': 'time'})
        annual  = annual.assign_coords(time=annual.time.values.astype(int))
        out.append(annual)
    return out


print('Reader functions defined.')

Reader functions defined.


## 4. Load data

In [ ]:

print('Loading MIROC-ES2H...')
miroc_HiLLA_pr  = read_miroc_pr(miroc_HiLLA_pr_files)
miroc_SAI_pr    = read_miroc_pr(miroc_SAI_pr_files)
miroc_ssp245_pr = read_miroc_pr(miroc_ssp245_pr_files)
print(f'  HiLLA {len(miroc_HiLLA_pr)}, SAI {len(miroc_SAI_pr)}, SSP245 {len(miroc_ssp245_pr)} members')


In [5]:
print('Loading CESM2-WACCM...')
cesm_HiLLA_pr  = read_pr_from_file_list(cesm_HiLLA_pr_files,  var='PRECT')
cesm_SAI_pr    = read_pr_from_file_list(cesm_SAI_pr_files,    var='PRECT')
cesm_ssp245_pr = read_pr_from_file_list(cesm_ssp245_pr_files, var='PRECT',
                                        open_kwargs={'anon': True})
cesm_HiLLA_pr  = [set_time_to_center_of_bounds(ds, time_bounds_name='time_bnds') for ds in cesm_HiLLA_pr]
cesm_SAI_pr    = [set_time_to_center_of_bounds(ds, time_bounds_name='time_bnds') for ds in cesm_SAI_pr]
cesm_ssp245_pr = [set_time_to_center_of_bounds(ds, time_bounds_name='time_bnds') for ds in cesm_ssp245_pr]
print(f'  HiLLA {len(cesm_HiLLA_pr)}, SAI {len(cesm_SAI_pr)}, SSP245 {len(cesm_ssp245_pr)} members')

print('Loading UKESM1-1...')
ukesm_HiLLA_pr  = read_pr_from_directory_paths(ukesm_HiLLA_pr_paths, time_slice=('2035', '2084'))
ukesm_ssp245_pr = read_pr_from_directory_paths(ukesm_ssp245_pr_paths)
print(f'  HiLLA {len(ukesm_HiLLA_pr)}, SSP245 {len(ukesm_ssp245_pr)} members (no SAI available)')

"""
print('Loading E3SMv3...')
e3sm_HiLLA_pr  = read_pr_from_file_list(e3sm_HiLLA_pr_files,  var='PRECT')
e3sm_ssp245_pr = read_pr_from_file_list(e3sm_ssp245_pr_files, var='PRECT')
e3sm_HiLLA_pr  = [set_time_to_center_of_bounds(ds, time_bounds_name='time_bnds') for ds in e3sm_HiLLA_pr]
e3sm_ssp245_pr = [set_time_to_center_of_bounds(ds, time_bounds_name='time_bnds') for ds in e3sm_ssp245_pr]
print(f'  HiLLA {len(e3sm_HiLLA_pr)}, SSP245 {len(e3sm_ssp245_pr)} members (no SAI available)')
"""

Loading CESM2-WACCM...


/tmp/ipykernel_167/4134932478.py:9: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset([f.open() for f in file_objects],
/tmp/ipykernel_167/4134932478.py:9: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset([f.open() for f in file_objects],
/tmp/ipykernel_167/4134932478.py:9

  HiLLA 3, SAI 3, SSP245 3 members
Loading UKESM1-1...


/tmp/ipykernel_167/4134932478.py:24: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset([f.open() for f in files],
/tmp/ipykernel_167/4134932478.py:24: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset([f.open() for f in files],
/tmp/ipykernel_167/4134932478.py:24: FutureWar

  HiLLA 3, SSP245 3 members (no SAI available)
Loading MIROC-ES2H...


FileNotFoundError: reflective-persistent-prod-large/MIROC-ES2H/G6-1.5K-SAI/Mon/pr_G6-1.5K-SAI_r01.nc

## 5. JJA monsoon precipitation timeseries

In [ ]:
print('Computing JJA monsoon-box means...')
jja_pr = {
    'CESM2-WACCM': {
        'SSP2-4.5':       jja_monsoon_ts(cesm_ssp245_pr, MPS_TO_MMDAY),
        'G6-1.5K-HiLLA':  jja_monsoon_ts(cesm_HiLLA_pr,  MPS_TO_MMDAY),
        'G6-1.5K-SAI':    jja_monsoon_ts(cesm_SAI_pr,    MPS_TO_MMDAY),
    },
    'UKESM1-1': {
        'SSP2-4.5':       jja_monsoon_ts(ukesm_ssp245_pr, KGM2S_TO_MMDAY),
        'G6-1.5K-HiLLA':  jja_monsoon_ts(ukesm_HiLLA_pr,  KGM2S_TO_MMDAY),
    },
    'MIROC-ES2H': {
        'SSP2-4.5':       jja_monsoon_ts(miroc_ssp245_pr, KGM2S_TO_MMDAY),
        'G6-1.5K-HiLLA':  jja_monsoon_ts(miroc_HiLLA_pr,  KGM2S_TO_MMDAY),
        'G6-1.5K-SAI':    jja_monsoon_ts(miroc_SAI_pr,    KGM2S_TO_MMDAY),
    },
    #'E3SMv3': {
    #    'SSP2-4.5':       jja_monsoon_ts(e3sm_ssp245_pr, MPS_TO_MMDAY),
    #    'G6-1.5K-HiLLA':  jja_monsoon_ts(e3sm_HiLLA_pr,  MPS_TO_MMDAY),
    #},
}

# Sanity check: SA monsoon-box JJA means should sit around ~4-8 mm/day.
# Values ~0.00005 or ~5000 indicate a unit-conversion problem (PRECT vs pr).
for m, d in jja_pr.items():
    for scen, ts_list in d.items():
        yrs = ts_list[0].time.values
        typ = float(ts_list[0].mean())
        print(f'  {m:<14} {scen:<20} {len(ts_list)} members  {yrs[0]}-{yrs[-1]}  '
              f'mean {typ:6.2f} mm/day')

## 6. Time of Emergence — cumulative paired t-test

One-sided *less*: emergence when the geoengineering scenario's monsoon precipitation is
significantly **weaker** than its paired SSP2-4.5 member over the cumulative window
[2035, Y] (monsoon weakening under aerosol/solar forcing).

In [ ]:
def cumulative_ttest_toe(scen_ts, ssp_ts, start_year=SAI_START, alpha=ALPHA,
                         min_n=MIN_N, alternative='less'):
    """Cumulative one-sided Welch's t-test: scen < ssp245 (monsoon weakening).
    scen_ts, ssp_ts: xarray DataArrays indexed by integer year.
    Returns first year Y >= start_year at which p < alpha, or None.
    """
    g_yrs, g_vals = scen_ts.time.values, scen_ts.values
    s_yrs, s_vals = ssp_ts.time.values,  ssp_ts.values
    g_yrs, g_vals = g_yrs[g_yrs >= start_year], g_vals[g_yrs >= start_year]
    s_yrs, s_vals = s_yrs[s_yrs >= start_year], s_vals[s_yrs >= start_year]
    end_year = int(min(g_yrs.max(), s_yrs.max()))
    for Y in range(start_year, end_year + 1):
        g_win, s_win = g_vals[g_yrs <= Y], s_vals[s_yrs <= Y]
        if min(len(g_win), len(s_win)) < min_n:
            continue
        _, p = stats.ttest_ind(g_win, s_win, equal_var=False, alternative=alternative)
        if p < alpha:
            return Y
    return None


def count_based_p66(member_toes):
    """k-th earliest member ToE, k = ceil(0.66 * N)."""
    valid = sorted(t for t in member_toes if t is not None)
    k = int(np.ceil(0.66 * len(member_toes)))
    return valid[k - 1] if len(valid) >= k else None


MODELS = ['CESM2-WACCM', 'UKESM1-1', 'MIROC-ES2H', 'E3SMv3']
SCENARIOS_BY_MODEL = {
    'CESM2-WACCM': ['G6-1.5K-HiLLA', 'G6-1.5K-SAI'],
    'UKESM1-1':    ['G6-1.5K-HiLLA'],
    'MIROC-ES2H':  ['G6-1.5K-HiLLA', 'G6-1.5K-SAI'],
    #'E3SMv3':      ['G6-1.5K-HiLLA'],
}

member_toes = {m: {} for m in MODELS}
for m in MODELS:
    ssp_list = jja_pr[m]['SSP2-4.5']
    for scen in SCENARIOS_BY_MODEL[m]:
        scen_list = jja_pr[m][scen]
        n = min(len(scen_list), len(ssp_list))
        member_toes[m][scen] = [
            cumulative_ttest_toe(scen_list[i], ssp_list[i]) for i in range(n)
        ]

print(f'{"Model":<14}{"Scenario":<20}{"Member ToEs":<50}{"p66 ToE":>8}')
print('-' * 92)
for m in MODELS:
    for scen in SCENARIOS_BY_MODEL[m]:
        toes = member_toes[m][scen]
        print(f'{m:<14}{scen:<20}{str(toes):<50}{str(count_based_p66(toes)):>8}')

# ── MIROC-ES2H bootstrap: all C(10,3) subsets ─────────────────────────────────
miroc_boot = {}
for scen in ['G6-1.5K-HiLLA', 'G6-1.5K-SAI']:
    toes10 = member_toes['MIROC-ES2H'][scen]
    boot   = [count_based_p66([toes10[i] for i in combo])
              for combo in combinations(range(10), 3)]
    miroc_boot[scen] = [b for b in boot if b is not None]
    if miroc_boot[scen]:
        p5, p25, p75, p95 = np.percentile(miroc_boot[scen], [5, 25, 75, 95])
        print(f'MIROC {scen}: 5/25/75/95th pct = {p5:.0f}/{p25:.0f}/{p75:.0f}/{p95:.0f} '
              f'({len(miroc_boot[scen])}/120 subsets emerged)')
    else:
        print(f'MIROC {scen}: no 3-member subset emerged')

## 7. Figure

In [ ]:
matplotlib.rcParams['font.size'] = 15

# ── Colours ───────────────────────────────────────────────────────────────────
GRAY        = '#636363'
GRAY_LIGHT  = '#D9D9D9'
HILLA_COL   = 'green'
HILLA_LIGHT = 'lightgreen'
SAI_COL     = 'purple'
SAI_LIGHT   = 'plum'

MODEL_COLORS = {
    'MIROC-ES2H':  '#0072B2',
    'CESM2-WACCM': '#009E73',
    'UKESM1-1':    '#E69F00',
    'E3SMv3':      '#CC3311',
}
MODEL_ORDER = ['MIROC-ES2H', 'CESM2-WACCM', 'UKESM1-1']#, 'E3SMv3']
MODEL_SIZE  = {'MIROC-ES2H': 190, 'CESM2-WACCM': 140, 'UKESM1-1': 95, 'E3SMv3': 55}
SCEN_MARKER = {'G6-1.5K-HiLLA': 's', 'G6-1.5K-SAI': 'o'}
SCEN_ORDER  = ['G6-1.5K-SAI', 'G6-1.5K-HiLLA']

SCEN_STYLE = [
    ('SSP2-4.5',       GRAY,      None),
    ('G6-1.5K-HiLLA',  HILLA_COL, None),
    ('G6-1.5K-SAI',    SAI_COL,   None),
]
MODEL_TS_ORDER = ['CESM2-WACCM', 'UKESM1-1', 'MIROC-ES2H']#, 'E3SMv3']

# ── Layout ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig,
                        width_ratios=[1, 1, 0.75],
                        wspace=0.32, hspace=0.38)

ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1], sharey=ax_a)
ax_c = fig.add_subplot(gs[1, 0], sharex=ax_a)
ax_d = fig.add_subplot(gs[1, 1], sharex=ax_b, sharey=ax_c)
ax_e = fig.add_subplot(gs[:, 2])

ax_ts     = [ax_a, ax_b, ax_c]#, ax_d]
ts_labels = ['(a)', '(b)', '(c)', '(d)']

# ── Timeseries panels ─────────────────────────────────────────────────────────
for ax, model, label in zip(ax_ts, MODEL_TS_ORDER, ts_labels):
    for scen, col, _ in SCEN_STYLE:
        if scen not in jja_pr[model]:
            continue
        ts_list = jja_pr[model][scen]
        toes_this = member_toes.get(model, {}).get(scen, [])

        for i, ts in enumerate(ts_list):
            yrs  = ts.time.values
            mask = (yrs >= PLOT_START) & (yrs <= PLOT_END)
            is_first = (i == 0)
            ax.plot(yrs[mask], ts.values[mask],
                    color=col, lw=0.8, alpha=0.55,
                    label=scen if is_first else '_')

            # individual member ToE vlines for geoengineering scenarios
            if scen != 'SSP2-4.5' and i < len(toes_this) and toes_this[i] is not None:
                ax.axvline(toes_this[i], color=col, lw=1.0, ls='--', alpha=0.65)

        # bold ensemble mean
        try:
            stacked = xr.concat(ts_list, dim='member', join='inner')
            yrs_c   = stacked.time.values
            mask_c  = (yrs_c >= PLOT_START) & (yrs_c <= PLOT_END)
            ax.plot(yrs_c[mask_c], stacked.values[:, mask_c].mean(axis=0),
                    color=col, lw=2.2)
        except Exception:
            pass

    ax.axvline(SAI_START, color='k', lw=0.8, ls='--', alpha=0.35)
    ax.set_title(f'{label} {model}', fontsize='large', pad=5)
    ax.set_xlim(PLOT_START, PLOT_END)
    ax.grid(ls='--', alpha=0.3)
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)

# y-labels and x-labels
ax_a.set_ylabel('SA monsoon JJA precip (mm day$^{-1}$)', fontsize='medium')
ax_c.set_ylabel('SA monsoon JJA precip (mm day$^{-1}$)', fontsize='medium')
ax_b.tick_params(labelleft=False)
#ax_d.tick_params(labelleft=False)
ax_a.tick_params(labelbottom=False)
ax_b.tick_params(labelbottom=False)
ax_c.set_xlabel('Year', fontsize='medium')
#ax_d.set_xlabel('Year', fontsize='medium')

# scenario legend in (a)
ax_a.legend(fontsize='small', loc='upper right', framealpha=0.88)

# ── Panel (e): ToE scatter ─────────────────────────────────────────────────────
for xi, scen in enumerate(SCEN_ORDER):
    for m in MODEL_ORDER:
        if scen not in SCENARIOS_BY_MODEL[m]:
            continue
        col = MODEL_COLORS[m]
        if m == 'MIROC-ES2H' and miroc_boot.get(scen):
            boot = miroc_boot[scen]
            p5, p95 = np.percentile(boot, [5, 95])
            bw = 0.08
            ax_e.add_patch(Rectangle(
                (xi - bw / 2, p5), bw, p95 - p5,
                facecolor=col, alpha=0.28, edgecolor=col, linewidth=1.2, zorder=2
            ))
        p66 = count_based_p66(member_toes[m][scen])
        if p66 is not None:
            face = 'none' if scen == 'G6-1.5K-HiLLA' else col
            ax_e.scatter([xi], [p66], marker=SCEN_MARKER[scen],
                         s=MODEL_SIZE[m], facecolors=face,
                         edgecolors=col, linewidths=1.8, zorder=4)

ax_e.set_xticks([0, 1])
ax_e.set_xticklabels(SCEN_ORDER, rotation=15, ha='right', fontsize='medium')
ax_e.set_xlim(-0.5, 1.5)
ax_e.set_ylabel('SA monsoon JJA precip Time of Emergence (year)', fontsize='medium')
ax_e.set_title('(e)', fontsize='large', pad=5)
ax_e.yaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))
ax_e.grid(ls='--', alpha=0.3, axis='y')
for sp in ('top', 'right'):
    ax_e.spines[sp].set_visible(False)

# combined legend for (e)
model_handles = [
    Line2D([0], [0], marker='o', color=MODEL_COLORS[m], lw=0,
           markerfacecolor=MODEL_COLORS[m], markersize=9, label=m)
    for m in MODEL_ORDER
]
scen_handles = [
    Line2D([0], [0], marker='s', color='0.3', lw=0, markerfacecolor='none',
           markeredgecolor='0.3', markersize=9, label='G6-1.5K-HiLLA'),
    Line2D([0], [0], marker='o', color='0.3', lw=0, markerfacecolor='0.3',
           markeredgecolor='0.3', markersize=9, label='G6-1.5K-SAI'),
]
ax_e.legend(handles=model_handles + scen_handles,
            fontsize='small', loc='upper left', framealpha=0.9)
plt.tight_layout()
plt.savefig('Figures/Monsoon_ToE_ttest.png', dpi=200, bbox_inches='tight')
plt.show()